# Pipeline de Unión de *datasets*

El paso final de toda la recolección de datos de la EEA y de *Open-Meteo* consiste en la creación de un *dataset* conjunto por país, donde se cruzan las métricas de calidad del aire con las variables climáticas. Unimos los *datasets* llevando a cabo los siguientes pasos:

1. **Sincronización de series**: dado que los sensores pueden tener ligeros desfases temporales, redondeamos todas las marcas de tiempo a la hora (*dt.floor("h")*) para crear una llave de unión consistente.

2. **Normalización de variables**: renombramos las columnas de valores genéricos a identificadores específicos que incluyen la unidad de medida (ej. PM10 ($ug/m^3$)), permitiendo tener múltiples contaminantes en un solo archivo.

3. **Creación de un *timeline* continuo**: generamos un índice horario ininterrumpido (*pd.date_range*). Esto es fundamental para modelos de series temporales, ya que permite identificar visualmente los periodos de inactividad de los sensores (datos faltantes/*NaN*).

In [ ]:
# ==============================================================================
# IMPORTACIÓN DE LIBRERÍAS
# ==============================================================================

import pandas as pd
from pathlib import Path
import os
import requests
import re
import time

In [ ]:
# ==============================================================================
# CONFIGURACIÓN DE RUTAS
# ------------------------------------------------------------------------------
FOLDER_POLLUTANTS = Path(r"C:\Users\Juanfran cd\Desktop\Máster_Ciencia_de_Datos_UA\TFM\datasets\archivos_csv")
FOLDER_CLIMA      = Path(r"C:\Users\Juanfran cd\Desktop\Máster_Ciencia_de_Datos_UA\TFM\datasets\archivos_clima")
FOLDER_FINAL      = Path(r"C:\Users\Juanfran cd\Desktop\Máster_Ciencia_de_Datos_UA\TFM\datasets\archivos_cont_clima")
FOLDER_FINAL.mkdir(exist_ok=True)

In [9]:
# Listamos las subcarpetas existentes para iterar por cada país.
paises = [f.name for f in FOLDER_POLLUTANTS.iterdir() if f.is_dir()]

for pais in paises:
    folder_poll = FOLDER_POLLUTANTS / pais
    folder_clima = FOLDER_CLIMA / pais

    # --------------------------------------------------------------------------
    # PROCESAMIENTO DE CONTAMINANTES (POLLUTANTS)
    # --------------------------------------------------------------------------

    dfs_poll = []
    for f in folder_poll.glob("*.csv"):
        df = pd.read_csv(f)

        # Normalización de strings y conversión a formato Datetime
        df["Start"] = df["Start"].astype(str).str.strip()
        df["End"]   = df["End"].astype(str).str.strip()
        df["Start"] = pd.to_datetime(df["Start"], errors="coerce")
        df["End"]   = pd.to_datetime(df["End"], errors="coerce")

        # Creamos una "llave temporal" redondeada a la hora para el cruce de datos
        df["Start_hour"] = df["Start"].dt.floor("h")

        # Renombramos la columna "Value" con el nombre del contaminante y su unidad
        # Ej: "Value" -> "NO2 (ug/m3)". Los contaminantes se demoninan con un índice numérico, no por su nombre.
        pollutant = df["Pollutant"].iloc[0]
        unit = df["Unit"].iloc[0]
        df = df.rename(columns={"Value": f"{pollutant} ({unit})"})
        
        # Seleccionamos solo las columnas necesarias para el merge
        dfs_poll.append(df[["Start", "End", "Start_hour", f"{pollutant} ({unit})"]])

    if not dfs_poll:
        continue

    # Unión horizontal (Merge) de todos los contaminantes del país
    df_poll = dfs_poll[0]
    for df_next in dfs_poll[1:]:
        df_poll = pd.merge(df_poll, df_next, on=["Start", "End", "Start_hour"], how="outer")

    # --------------------------------------------------------------------------
    # PROCESAMIENTO DE DATOS CLIMÁTICOS
    # --------------------------------------------------------------------------
    dfs_clima = []
    for f in folder_clima.glob("*.csv"):
        df = pd.read_csv(f)
        # Sincronizamos la columna de tiempo de Open-Meteo con nuestra llave 'Start_hour'
        df["Start_hour"] = pd.to_datetime(df["time"].astype(str).str.strip())
        df = df.drop(columns=["time"])
        dfs_clima.append(df)

    if dfs_clima:
        df_clima = dfs_clima[0]
        for df_next in dfs_clima[1:]:
            df_clima = pd.merge(df_clima, df_next, on="Start_hour", how="outer")
    else:
        df_clima = pd.DataFrame()
    
    # --------------------------------------------------------------------------
    # CONSTRUCCIÓN DEL TIMELINE FINAL
    # --------------------------------------------------------------------------
    # Generamos un rango horario continuo sin huecos desde el inicio al fin de los datos.
    # Esto asegura que si falta una medición, quede como NaN en lugar de desaparecer la fila.
    min_start = df_poll["Start"].min()
    max_end = df_poll["End"].max()
    idx = pd.date_range(start=min_start.floor("h"), end=max_end.ceil("h"), freq="h")

    df_final = pd.DataFrame({"Start_hour": idx})
    
    # Unimos el timeline con contaminantes y luego con clima.
    df_final = pd.merge(df_final, df_poll, on="Start_hour", how="left")
    if not df_clima.empty:
        df_final = pd.merge(df_final, df_clima, on="Start_hour", how="left")

    # Reconstruimos Start y End para que sean consistentes y eliminamos llaves temporales.
    df_final["Start"] = df_final["Start_hour"]
    df_final["End"]   = df_final["Start_hour"] + pd.Timedelta(hours=1)
    df_final = df_final.drop(columns=["Start_hour"])

    # Reordenamos columnas para que las fechas siempre aparezcan al principio.
    cols = ["Start", "End"] + [c for c in df_final.columns if c not in ["Start", "End"]]
    df_final = df_final[cols]

    output_file = FOLDER_FINAL / f"{pais}.csv"
    df_final.to_csv(output_file, index=False)

print("\n✅ Integración completada: Contaminantes y Clima unidos por país.")


✅ Integración completada: Contaminantes y Clima unidos por país.


Una vez unidos todos los datasets, vamos a cambiar la codificación que se le da a los compuestos o elementos presentes en el aire por su formulación química (su nombre real).

In [10]:
# ---------------------------------------------------------
# OBTENER DICCIONARIO USANDO "PK" (El número simple)
# ---------------------------------------------------------
# Este bloque conecta con la base de datos maestra de la Agencia Europea de Medio Ambiente para obtener la relación entre los códigos numéricos y los nombres de los gases.

try:
    # URL de la API que contiene el catálogo de todos los contaminantes (648 compuestos)
    url_pollutants = "https://eeadmz1-downloads-api-appservice.azurewebsites.net/Pollutant"
    
    # Realizamos la petición GET y convertimos la respuesta JSON en una lista de Python
    data = requests.get(url_pollutants).json()
    
    # EXPLICACIÓN DEL DICCIONARIO:
    # Creamos un diccionario de búsqueda rápida (comprensión de diccionario).
    # Usamos "pk" como clave porque es el número entero simple (1, 8, 5...) que aparece en tus CSV.
    # Usamos "notation" como valor porque es la abreviatura química (SO2, NO2, PM10...).
    # El resultado es un mapa tipo: {"8": "NO2", "5": "PM10", "7": "O3", ...}
    diccionario_aire = {str(p['pk']): p['notation'] for p in data}


except Exception as e:
    # Si la API falla o no hay internet, el programa se detiene para evitar errores en cadena
    print(f"   [ERROR] No se pudo conectar con la API: {e}")
    exit()

In [11]:
# ---------------------------------------------------------
# FUNCIÓN DE RENOMBRADO ADAPTADA AlFORMATO
# ---------------------------------------------------------
# Esta función toma el nombre de una columna y lo transforma si detecta un ID numérico.

def traducir_columna(col_name):
    # EXPRESIÓN REGULAR (RegEx):
    # ^(\d+)  -> Busca uno o más dígitos al inicio de la cadena.
    # (.*)    -> Captura todo lo que venga después (unidades, paréntesis, etc.).
    # .strip() -> Elimina espacios en blanco accidentales al principio o final.
    match = re.match(r"^(\d+)(.*)$", col_name.strip())
    
    if match:
        # Extraemos el número (ej: "8") y el resto del texto (ej: " (ug.m-3)")
        numero_pk = match.group(1)
        resto = match.group(2)
        
        # Si el número extraído existe en nuestro diccionario de la API...
        if numero_pk in diccionario_aire:
            # Reemplazamos el número por su nombre químico pero mantenemos las unidades originales
            # Resultado ejemplo: "8 (ug.m-3)" -> "NO2 (ug.m-3)"
            return f"{diccionario_aire[numero_pk]}{resto}"
            
    # Si la columna no empieza por un número (ej: "Start", "temperature"), se devuelve tal cual
    return col_name

In [13]:
# ---------------------------------------------------------
# PROCESAMIENTO
# ---------------------------------------------------------
# Listamos todos los archivos CSV dentro de la carpeta final del proyecto
archivos_csv = [f for f in os.listdir(FOLDER_FINAL) if f.endswith('.csv')]

for archivo in archivos_csv:
    # Construimos la ruta completa al archivo
    ruta_completa = os.path.join(FOLDER_FINAL, archivo)
    
    try:
        # Leemos el archivo CSV completo en un DataFrame de Pandas
        # low_memory=False se usa para evitar advertencias si el archivo tiene muchas filas
        df = pd.read_csv(ruta_completa, low_memory=False)
        
        # Guardamos la lista original de nombres de columnas para comparar después
        viejas_cols = list(df.columns)
        
        # Aplicamos la función de traducción a cada una de las columnas
        nuevas_cols = [traducir_columna(c) for c in viejas_cols]
        
        # Solo guardamos el archivo si realmente hemos cambiado algún nombre de columna
        if viejas_cols != nuevas_cols:
            df.columns = nuevas_cols
            
            # Guardamos el archivo sobreescribiendo el original.
            # index=False evita que se cree una columna extra de números al principio.
            # encoding='utf-8-sig' asegura que símbolos como el grado (°) se lean bien en Excel.
            df.to_csv(ruta_completa, index=False, encoding='utf-8-sig')
            print(f"   [MODIFICADO] {archivo}")
        else:
            # Si el archivo ya estaba traducido o no tenía IDs, informamos al usuario
            print(f"   [AVISO] No se encontraron IDs para cambiar en: {archivo}")
            
    except Exception as e:
        # Si el archivo está abierto en Excel o está corrupto, saltará este error
        print(f"   [ERROR] Fallo en {archivo}: {e}")

print("✅ PROCESO FINALIZADO.")

   [MODIFICADO] IE.csv
✅ PROCESO FINALIZADO.
